# ML Pipeline — Train + Evaluate

This notebook runs the minimal end-to-end ML pipeline for TRI-GEN: clean raw data, build features, train forecasters and anomaly detectors, then evaluate the results.

It skips exploratory inspection and EDA report generation so you can run training and validation quickly.

> Note: the current pipeline uses regression metrics for forecasters (MAE/RMSE/MAPE/sMAPE). There is no F1 score for the regression models unless labeled anomaly ground truth is added separately.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

def find_ml_pipeline_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]

    for base in candidates:
        if (base / 'PIPELINE.md').exists() and (base / 'src').exists():
            return base
        nested = base / 'apps' / 'ml-pipeline'
        if (nested / 'PIPELINE.md').exists() and (nested / 'src').exists():
            return nested

    raise RuntimeError('Could not find apps/ml-pipeline. Open this notebook from the repo or ml-pipeline folder.')

ML_PIPELINE_DIR = find_ml_pipeline_dir()
os.chdir(ML_PIPELINE_DIR)

print(f'Using ML pipeline directory: {ML_PIPELINE_DIR}')
print(f'Python executable: {sys.executable}')

In [ ]:
INSTALL_REQUIREMENTS = True

if INSTALL_REQUIREMENTS:
    print('Installing requirements into the active kernel environment...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
else:
    print('Skipping requirements install. Set INSTALL_REQUIREMENTS = True if dependencies are missing.')

## Pipeline Runner

This cell defines the minimal pipeline stages used for training and evaluation.

In [ ]:
import time
from pathlib import Path

STAGES = [
    {
        'id': 1,
        'name': 'Clean TRI-GEN raw data',
        'module': 'src.data.clean_tri_gen',
        'outputs': ['../../data/processed/tri-gen/long.parquet', '../../data/processed/tri-gen/_build_manifest.json'],
    },
    {
        'id': 2,
        'name': 'Build lag + rolling features',
        'module': 'src.features.build_features',
        'outputs': ['../../data/processed/tri-gen/features.parquet', '../../data/processed/tri-gen/_features_manifest.json'],
    },
    {
        'id': 3,
        'name': 'Train forecasters',
        'module': 'src.training.train_forecaster',
        'outputs': ['../../checkpoints/forecaster/_summary.csv'],
    },
    {
        'id': 4,
        'name': 'Train anomaly detectors',
        'module': 'src.training.train_anomaly',
        'outputs': ['../../checkpoints/anomaly/_summary.csv'],
    },
    {
        'id': 5,
        'name': 'Evaluate trained models',
        'module': 'src.evaluation.evaluate',
        'outputs': ['reports/eval/index.html', 'reports/eval/forecaster_leaderboard.csv'],
    },
]


def run_command(command: list[str], cwd: Path = ML_PIPELINE_DIR) -> None:
    print('Running:', ' '.join(command))
    process = subprocess.Popen(
        command,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    rc = process.wait()
    if rc != 0:
        raise RuntimeError(f'Command failed with exit code {rc}')


def run_stage(stage: dict) -> None:
    print('\n' + '='*80)
    print(f"Stage {stage['id']}: {stage['name']}")
    print('='*80)
    start = time.time()
    run_command([sys.executable, '-m', stage['module']])
    elapsed = time.time() - start
    print(f'\nStage completed in {elapsed:.1f}s')
    for rel in stage.get('outputs', []):
        path = (ML_PIPELINE_DIR / rel).resolve()
        print(f'  [{"ok" if path.exists() else "missing"}] {path}')


def run_pipeline(start_stage: int = 1, end_stage: int = 5) -> None:
    selected = [s for s in STAGES if start_stage <= s['id'] <= end_stage]
    if not selected:
        raise ValueError(f'No stages selected for {start_stage}..{end_stage}')
    for stage in selected:
        run_stage(stage)

for stage in STAGES:
    print(f"{stage['id']}. {stage['name']} -> python -m {stage['module']}")


## Run the full training + evaluation pipeline

Execute the following cell to run all necessary stages from raw data through final evaluation.

In [ ]:
run_pipeline(start_stage=1, end_stage=5)

## Inspect outputs

The following cell reports the main generated artifacts.

In [ ]:
from pathlib import Path

paths = [
    ML_PIPELINE_DIR / '../../data/processed/tri-gen/long.parquet',
    ML_PIPELINE_DIR / '../../data/processed/tri-gen/features.parquet',
    ML_PIPELINE_DIR / '../../checkpoints/forecaster/_summary.csv',
    ML_PIPELINE_DIR / '../../checkpoints/anomaly/_summary.csv',
    ML_PIPELINE_DIR / 'reports/eval/index.html',
    ML_PIPELINE_DIR / 'reports/eval/forecaster_leaderboard.csv',
]
for p in paths:
    status = 'ok' if p.exists() else 'missing'
    print(f'{status:7s} {p.resolve()}')
